# LB 25.36727 | QCC analysis toolkit: exact simulator, cost anatomy, what works and what fails

**This notebook submits the same `submission.zip` as our solution notebook**
([LB 25.36727 | DQI readout windows on bb4287e7](https://www.kaggle.com/code/asteriospanoras/lb-25-36727-dqi-readout-windows-on-bb4287e7)),
byte for byte, so it carries the same public score. Its purpose is different: it is an **open toolkit and a lab
notebook** for the three tasks.

**What you get**
1. `kmsim`: a ~150-line numpy re-implementation of the kickmix subset these circuits use. It reproduces the
   official invsqrt verifier exactly (same mean Toffolis, same worst error/tolerance) in ~2 s for 8k shots, and
   you can stop it after any operation to inspect every wire. Use it to test circuit surgery before the 15-minute
   official run.
2. **Exchange rates** (what one qubit, Toffoli or rotation is worth per task), the **public frontier over time**,
   and **landscapes** of every public circuit in qubits x Toffolis with lines of equal cost.
3. Cost anatomy of the best public circuit for each task: where the Toffolis, qubits and rotations go.
4. An **ablation study with the official verifier** of our DQI change, and **how lucky a board draw is** (the score
   distribution of this decoder line, which matters for the private leaderboard).
5. What works and what does not, **with the measurements**: the DQI readout-window identity (and a proof by
   exhaustion), why merging the two invsqrt phase streams is blocked, why splicing all 50 public 62-qubit invsqrt
   circuits finds nothing beyond HW-25, why transplanting Heliosli's retained product alone breaks the phase.
6. Open problems at the end. **Reviews, corrections and ideas are very welcome in the comments**: if you try one of
   them, say what happened, negative results included.

Everything runs on the Kaggle CPU image with the competition data. All circuits are public (credits at the end).

In [ ]:
import sys, os, pathlib, zipfile, json, time, re, hashlib, shutil, math, difflib, heapq, collections
import numpy as np
import matplotlib.pyplot as plt
T0 = time.time()
def log(*a): print(f"[{(time.time()-T0)/60:5.1f} min]", *a, flush=True)
COMP = pathlib.Path("/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation")
KM = COMP / "kickmix"; VER = COMP / "verifiers"
sys.path.insert(0, str(KM))
import kickmix as km
inp = pathlib.Path("/kaggle/input"); WK = pathlib.Path("/kaggle/working"); TMP = pathlib.Path("/tmp/a"); TMP.mkdir(exist_ok=True)
sha = lambda p: hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()
# Same submission as the solution notebook: copy its output byte for byte.
subzip = [p for p in inp.rglob("submission.zip") if "readout-windows" in str(p)][0]
shutil.copy(subzip, WK / "submission.zip")
receipt = json.load(open(subzip.parent / "receipt.json"))
with zipfile.ZipFile(WK / "submission.zip") as z: z.extractall(TMP / "sub")
log("submission.zip copied", sha(WK / "submission.zip")[:12], {k: v[:8] for k, v in receipt["sha256"].items() if "angle" not in k})
# Chart style (validated categorical palette, recessive axes)
C = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
plt.rcParams.update({"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": "#b9b8b3",
                     "axes.labelcolor": "#52514e", "xtick.color": "#52514e", "ytick.color": "#52514e", "axes.grid": True,
                     "grid.color": "#e8e7e3", "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "lines.linewidth": 2})

## 1. The score in one table
Cost per task from the official verifiers (run in the solution notebook), and the improvement over the hosts'
reference. The score is the geometric mean of the three ratios, so a 1% cost cut on any task is worth the same.

In [ ]:
REF = {"invsqrt": 2076774.23, "dqi": 24439463583672.93, "dfthcsos": 3355599588.32}
v = receipt["verify"]
rows = {
    "invsqrt": (v["invsqrt"]["num_qubits"], v["invsqrt"]["toffoli_count"], v["invsqrt"]["score"]),
    "dqi": (v["dqi"]["num_qubits"], v["dqi"]["toffoli_count"], v["dqi"]["score"] / v["dqi"]["repetition_factor"] * 1.1912604648115832),
    "dfthcsos": (v["dfthcsos"]["num_qubits"], v["dfthcsos"]["toffoli_count"], v["dfthcsos"]["score"]),
}
print(f"{'task':9s} {'Q':>5s} {'Toffolis/shot':>14s} {'cost':>18s} {'ratio':>9s}")
ratios = {}
for k, (q, t, c) in rows.items():
    ratios[k] = REF[k] / c
    print(f"{k:9s} {q:5d} {t:14,.1f} {c:18,.2f} {ratios[k]:9.4f}")
print("geometric mean (DQI at zero failed shots):", round(math.prod(ratios.values()) ** (1 / 3), 5))
fig, ax = plt.subplots(figsize=(6.5, 2.6))
ax.barh(list(ratios), list(ratios.values()), color=C[0], height=0.5)
for i, (k, r) in enumerate(ratios.items()): ax.text(r, i, f"  {r:.2f}x", va="center", color="#0b0b0b")
ax.set_title("Improvement over the reference circuit, per task"); ax.set_xlabel("reference cost / cost"); ax.invert_yaxis()
plt.tight_layout(); plt.show()
print("""Reading: chemistry has the smallest ratio, but about half of its Toffolis and 303 of its 427 qubits come from the
verifier's cost model of the state preparations (section 5), not from the submitted circuits.""")

### Exchange rates: what one qubit, one Toffoli or one rotation is worth
Each task's cost is a product, so a relative cut in any factor moves the score by the same relative amount
(divided by 3 in the geometric mean). The table converts one unit of each resource into score points at the
current frontier, and into Toffolis of the same task. It is the first thing to check before any circuit surgery.

In [ ]:
gm = math.prod(ratios.values()) ** (1 / 3)
inv_C = v["invsqrt"]["toffoli_equivalent"]; inv_Q = v["invsqrt"]["num_qubits"]
dqi_T = v["dqi"]["toffoli_count"]; dqi_Q = v["dqi"]["num_qubits"]
ch_T = v["dfthcsos"]["toffoli_count"]; ch_Q = v["dfthcsos"]["num_qubits"]; ch_l = v["dfthcsos"]["lambda_eff"]
rot1 = (0.56 * math.log2(1 / v["invsqrt"]["rotation_eps"]) + 5.3) / 4
xr = [  # (task, resource, relative cost change of removing one unit, in Toffolis of that task)
    ("invsqrt", "1 qubit", 1 / inv_Q, inv_C / inv_Q),
    ("invsqrt", "1 Toffoli", 1 / inv_C, 1.0),
    ("invsqrt", "1 rotation", rot1 / inv_C, rot1),
    ("dqi", "1 qubit (Q squared)", 1 - ((dqi_Q - 1) / dqi_Q) ** 2, (1 - ((dqi_Q - 1) / dqi_Q) ** 2) * dqi_T),
    ("dqi", "1,000 Toffolis", 1000 / dqi_T, 1000.0),
    ("dfthcsos", "1 qubit", 1 / ch_Q, ch_T / ch_Q),
    ("dfthcsos", "1 Toffoli", 1 / ch_T, 1.0),
    ("dfthcsos", "0.01 Ha of lambda_eff", 0.01 / ch_l, 0.01 / ch_l * ch_T),
]
print(f"{'task':9s} {'resource':24s} {'cost change':>12s} {'score points':>13s} {'= Toffolis':>11s}")
for t_, res_, rel, tof in xr:
    pts = gm * ((1 / (1 - rel)) ** (1 / 3) - 1)
    print(f"{t_:9s} {res_:24s} {-100 * rel:11.4f}% {pts:+13.5f} {tof:11.1f}")
print("""Reading: an invsqrt qubit is worth ~24 Toffolis, a DQI qubit ~880 Toffolis, a chemistry qubit ~35 Toffolis.
On this board 0.001 points is ~0.012% of one task's cost: the public step from 25.36623 to 25.36727 was 48 Toffolis
out of 394,165 in DQI.""")

### The public frontier over time
manwithacat's *VQCC public circuits* dataset records, for every public circuit, its release time and its cost under
the official verifier. The best public ratio per task by day shows where the community found its gains.

In [ ]:
import csv
man = list(csv.DictReader(open(next(inp.rglob("manifest.csv")))))
def fnum(x):
    try: return float(x)
    except (TypeError, ValueError): return math.nan
task_key = {"invsqrt": "invsqrt", "dqi": "dqi", "chemistry": "dfthcsos"}
series = {}
for t_, key in task_key.items():
    pts = sorted((r["release_date"][:16], fnum(r["dqi_zero_failure_cost"]) if t_ == "dqi" and r["dqi_zero_failure_cost"] else fnum(r["local_task_score"])) for r in man if r["task"] == t_)
    best, out = math.inf, []
    for d, c in pts:
        if c == c and c < best: best = c; out.append((np.datetime64(d.replace(" ", "T")), REF[key] / c))
    series[t_] = out
fig, ax = plt.subplots(figsize=(9, 3.2))
for (t_, out), col in zip(series.items(), C):
    d, r = zip(*out); ax.step(d, r, where="post", color=col, label=t_); ax.scatter(d, r, s=14, color=col, zorder=3)
    ax.text(d[-1], r[-1], f"  {r[-1]:.1f}x", color="#0b0b0b", va="center", fontsize=9)
ax.set_yscale("log"); ax.set_ylabel("reference / best public cost")
from matplotlib.ticker import FixedLocator, FormatStrFormatter
ax.yaxis.set_major_locator(FixedLocator([5, 10, 20, 40, 80])); ax.yaxis.set_major_formatter(FormatStrFormatter("%gx")); ax.yaxis.set_minor_locator(FixedLocator([])); ax.set_title("Best public circuit per task, by release time")
ax.legend(frameon=False, loc="upper left"); fig.autofmt_xdate(); plt.tight_layout(); plt.show()
for t_, out in series.items():
    print(f"{t_:9s} {len(out):3d} public records; last at {str(out[-1][0])[:16]}: {out[-1][1]:.3f}x")

## 2. `kmsim`: an exact numpy simulator for these circuits
Semantics from kickmix's `op_type.h` (open source in Qualtran): `HMR` = Hadamard, measure, reset (random outcome m,
phase (-1)^(m v), qubit -> 0, bit <- m); `R` = reset (an error if the qubit was 1); an operation runs only if every
pushed condition bit and its own `if b` bit are 1; `Z_POW` angles are half-turns. The phase is tracked in units of
2^-64 turns, like the verifier. Copy this cell to use it elsewhere (Apache-2.0).

In [ ]:
"""kmsim: pure-Python/numpy kickmix-subset simulator for analysis (not the official verifier).

Semantics from kickmix op_type.h: HMR = Hadamard+measure+reset (outcome random m, phase (-1)^(m*v), qubit -> 0,
bit <- m); R = reset (error if qubit was 1); conditions: op runs only if all pushed condition bits are 1 and its own
`if b` bit is 1. Phase tracked in units of 2^-64 turns (uint64), Z_POW angles are half-turns.
"""
from fractions import Fraction
import numpy as np

MASK = (1 << 64) - 1
TOFF = {"CCX", "CCZ", "CSWAP"}


def half_turns_units(s):
    f = Fraction(s)  # exact decimal
    return int(round(f * (1 << 63))) & MASK  # half-turn = 2^63 units


def parse(path):
    ops = []
    regs = {}
    for line in open(path):
        t = line.split()
        if not t:
            continue
        if t[0] == "APPEND_TO_REGISTER":
            regs.setdefault(t[2], []).append(int(t[1][1:])); continue
        if t[0] == "REGISTER":
            continue
        cond = None
        if "if" in t:
            i = t.index("if"); cond = int(t[i + 1][1:]); t = t[:i]
        name = t[0]; args = t[1:]
        qs = [int(a[1:]) for a in args if a.startswith("q")]
        bs = [int(a[1:]) for a in args if a.startswith("b")]
        ang = None
        if name == "Z_POW":
            ang = half_turns_units(args[1])
        ops.append((name, qs, bs, cond, ang))
    return ops, regs


class Sim:
    def __init__(self, ops, nq, nb, batch, seed=0):
        self.ops = ops; self.B = batch
        self.q = np.zeros((nq, batch), dtype=bool)
        self.b = np.zeros((nb, batch), dtype=bool)
        self.ph = np.zeros(batch, dtype=np.uint64)
        self.rng = np.random.default_rng(seed)
        self.toff = np.zeros(batch, dtype=np.int64)
        self.rot = np.zeros(batch, dtype=np.int64)
        self.bad_reset = np.zeros(batch, dtype=bool)

    def run(self, start=0, stop=None, hook=None):
        stack = getattr(self, "stack", [])
        q, b = self.q, self.b
        ones = np.ones(self.B, dtype=bool)
        for k in range(start, len(self.ops) if stop is None else stop):
            name, qs, bs, cond, ang = self.ops[k]
            if name == "PUSH_CONDITION":
                stack.append(cond); continue
            if name == "POP_CONDITION":
                stack.pop(); continue
            act = ones
            if stack or cond is not None:
                act = ones.copy()
                for c in stack: act &= b[c]
                if cond is not None: act &= b[cond]
            if name == "CX":
                q[qs[1]] ^= q[qs[0]] & act
            elif name == "CZ":
                self._neg(q[qs[0]] & q[qs[1]] & act)
            elif name == "Z":
                self._neg(q[qs[0]] & act)
            elif name == "X":
                q[qs[0]] ^= act
            elif name == "CCX":
                q[qs[2]] ^= q[qs[0]] & q[qs[1]] & act; self.toff += act
            elif name == "CCZ":
                self._neg(q[qs[0]] & q[qs[1]] & q[qs[2]] & act); self.toff += act
            elif name == "SWAP":
                a, c = q[qs[0]].copy(), q[qs[1]].copy()
                q[qs[0]] = np.where(act, c, a); q[qs[1]] = np.where(act, a, c)
            elif name == "CSWAP":
                m = act & q[qs[0]]
                a, c = q[qs[1]].copy(), q[qs[2]].copy()
                q[qs[1]] = np.where(m, c, a); q[qs[2]] = np.where(m, a, c); self.toff += act
            elif name == "NEG":
                self._neg(act)
            elif name == "HMR":
                m = self.rng.random(self.B) < 0.5
                self._neg(m & q[qs[0]] & act)
                q[qs[0]] &= ~act
                b[bs[0]] = np.where(act, m, b[bs[0]])
            elif name == "R":
                self.bad_reset |= q[qs[0]] & act
                q[qs[0]] &= ~act
            elif name == "Z_POW":
                m = q[qs[0]] & act
                self.ph = self.ph + np.where(m, np.uint64(ang), np.uint64(0)); self.rot += act
            elif name == "BIT_STORE0":
                b[bs[0]] &= ~act
            elif name == "BIT_STORE1":
                b[bs[0]] |= act
            elif name == "BIT_INVERT":
                b[bs[0]] ^= act
            elif name in ("INFO_TAG", "DEBUG_PRINT"):
                pass
            else:
                raise ValueError(name)
            if hook is not None:
                hook(k, self)
        self.stack = stack

    def _neg(self, m):
        self.ph = self.ph + np.where(m, np.uint64(1 << 63), np.uint64(0))


def target_and_tol(xs, precision_bits=24):
    import math
    rs = [math.isqrt((1 << 130) // int(x)) for x in xs]
    target = np.array([((r + 1) >> 1) & MASK for r in rs], dtype=np.uint64)
    tol = np.array([min(r >> (precision_bits + 1), 1 << 63) for r in rs], dtype=np.uint64)
    return target, tol


def check(path, xs, seed=0, ops_regs=None):
    ops, regs = ops_regs or parse(path)
    nq = 1 + max(max(o[1]) for o in ops if o[1])
    nb = 1 + max((max(o[2]) for o in ops if o[2]), default=0)
    nb = max(nb, 1 + max((o[3] for o in ops if o[3] is not None), default=0))
    s = Sim(ops, nq, nb, len(xs), seed)
    xq = regs["r0"]
    for i, qi in enumerate(xq):
        s.q[qi] = (xs >> i) & 1 == 1
    s.run()
    xr = sum((s.q[qi].astype(np.uint64) << np.uint64(i)) for i, qi in enumerate(xq))
    restored = np.array_equal(xr, xs.astype(np.uint64))
    scratch = [i for i in range(nq) if i not in xq]
    dirty = s.q[scratch].any(axis=0)
    target, tol = target_and_tol(xs)
    err = (s.ph - target).view(np.int64)
    used = np.abs(err).astype(np.float64)
    return dict(nq=nq, restored=restored, dirty=int(dirty.sum()), bad_reset=int(s.bad_reset.sum()),
                worst_ratio=float((used / tol.astype(np.float64)).max()), toff_mean=float(s.toff.mean()),
                rot_mean=float(s.rot.mean())), s

**Validation against the official verifier.** Export the invsqrt circuit to text, run kmsim on all x up to 4,000
plus 12,000 random x and the known worst cases, and compare with the official numbers from the solution notebook.

In [ ]:
def to_kmx(src, dst):
    km.Circuit.from_file(src, format=pathlib.Path(src).suffix[1:]).to_file(dst, format="kmx"); return dst
INV = to_kmx(TMP / "sub" / "invsqrt.kmb", TMP / "invsqrt.kmx")
BASE = to_kmx(COMP / "sample_submission" / "invsqrt.kmb", TMP / "invsqrt_reference.kmx")
rng = np.random.default_rng(0)
xs = np.concatenate([np.arange(1, 4001), rng.integers(1, 3133453, 12000), np.array([161, 221, 644, 3099851, 3133452])]).astype(np.int64)
t = time.time(); r_inv, s_inv = check(INV, xs, seed=1); dt = time.time() - t
print("kmsim  :", {k: r_inv[k] for k in ("nq", "restored", "dirty", "worst_ratio", "toff_mean")}, f"({dt:.1f} s for {len(xs):,} shots)")
print("verifier:", {k: v["invsqrt"].get(k) for k in ("num_qubits", "worst_error_over_tolerance", "toffoli_count")})
r_base, _ = check(BASE, xs[:4000], seed=1)
print("reference circuit:", {k: r_base[k] for k in ("nq", "worst_ratio", "toff_mean")})

## 3. invsqrt anatomy: where the 1,514 Toffoli-equivalents go
The verifier's cost is Q x C with C = mean Toffolis + T/4, and every non-Clifford rotation priced at
0.56 log2(1/eps) + 5.3 T, where eps splits the leftover error margin evenly across rotations. Below: cumulative
Toffolis and qubits in use (non-zero on at least one shot) along the circuit, with the rotations marked.

In [ ]:
ops, regs = parse(INV)
xs_a = np.concatenate([np.arange(1, 1501), rng.integers(1, 3133453, 2500)]).astype(np.int64)
s = Sim(ops, 62, 1 + max(max((max(o[2]) for o in ops if o[2]), default=0), max((o[3] for o in ops if o[3] is not None), default=0)), len(xs_a), 3)
for i, qi in enumerate(regs["r0"]): s.q[qi] = (xs_a >> i) & 1 == 1
cum_t = np.zeros(len(ops)); live = np.zeros(len(ops), dtype=int)
def hook(k, sim):
    cum_t[k] = sim.toff.mean()
    if k % 5 == 0: live[k] = int(sim.q.any(axis=1).sum())
s.run(hook=hook)
live = np.where(live > 0, live, np.nan).astype(float)
rot_idx = [k for k, o in enumerate(ops) if o[0] == "Z_POW" and o[4] % (1 << 61) != 0]
fig, axs = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
axs[0].plot(cum_t, color=C[0]); axs[0].set_title("invsqrt: cumulative mean Toffolis per shot"); axs[0].set_ylabel("Toffolis")
for k in rot_idx: axs[0].axvline(k, color=C[1], lw=0.6, alpha=0.6)
axs[0].text(rot_idx[0], cum_t[-1] * 0.9, " rotations (orange)", color="#52514e")
ok = ~np.isnan(live); axs[1].plot(np.flatnonzero(ok), live[ok], color=C[2], lw=1.2)
axs[1].set_title("qubits holding data (out of 62)"); axs[1].set_ylabel("qubits"); axs[1].set_xlabel("operation index")
plt.tight_layout(); plt.show()
n_rot = len(rot_idx); rot_cost = n_rot * (0.56 * math.log2(1 / v["invsqrt"]["rotation_eps"]) + 5.3) / 4
print(f"{n_rot} rotations at eps={v['invsqrt']['rotation_eps']:.2e}: {rot_cost:.0f} Toffoli-eq "
      f"= {100 * rot_cost / v['invsqrt']['toffoli_equivalent']:.1f}% of C; one qubit = {100 / 62:.2f}% of the score")

### The two phase streams
Every rotation acts on one helper wire (`q28`): a multiplexer (the shifter, 1-4 Toffolis) routes one bit of the
value into it, the rotation fires, and the helper is measured away. Stage 1 phases a table value, stage 2 the
interpolation product. Their weights overlap on 8 positions.

In [ ]:
info = []
for k in rot_idx:
    j = k - 1
    while j > 0 and not (ops[j][0] in ("HMR", "Z_POW") and ops[j][1] == [28]): j -= 1
    seg = ops[j + 1:k]
    src = next((o[1][0] for o in seg if o[0] == "CX" and o[1][1] == 28), None)
    info.append((k, ops[k][4] / 2**63, sum(o[0] in TOFF for o in seg), src))
w = np.log2([i[1] for i in info]); stage = np.array([0 if k < 12000 else 1 for k, *_ in info])
fig, ax = plt.subplots(figsize=(9, 2.8))
for st, col, lab in ((0, C[0], "stage 1 (table value)"), (1, C[1], "stage 2 (interpolation product)")):
    m = stage == st; ax.scatter(w[m], [st] * m.sum(), s=40, color=col, label=lab, edgecolor="#fcfcfb", linewidth=1.5, zorder=3)
ax.set_yticks([0, 1]); ax.set_yticklabels(["stage 1", "stage 2"]); ax.set_xlabel("rotation weight, log2(half-turns)")
ax.set_title("Rotation weights: 8 positions are paid twice"); ax.legend(loc="lower right", frameon=False); ax.set_ylim(-0.6, 1.6)
plt.tight_layout(); plt.show()
ov = sorted(set(np.round(w[stage == 0])) & set(np.round(w[stage == 1])))
print("overlapping weights (log2 half-turns):", ov)
# Is the stage-1 source of each overlapping bit still intact when stage 2 runs?
s2 = Sim(ops, 62, s.b.shape[0], 512, 5)
xs_b = rng.integers(1, 3133453, 512).astype(np.int64)
for i, qi in enumerate(regs["r0"]): s2.q[qi] = (xs_b >> i) & 1 == 1
first_stage2 = min(k for k, *_ in info if k >= 12000)
srcs = [src for (k, wt, nt, src), st in zip(info, stage) if st == 0 and src is not None and round(math.log2(wt)) in ov]
rot_src = {i[0]: i[3] for i, st in zip(info, stage) if st == 0 and i[3] in srcs}
snap = {}
def hook2(k, sim):
    if k in rot_src: snap[k] = sim.q[rot_src[k]].copy()
s2.run(0, first_stage2, hook=hook2)
alive = {f"q{rot_src[k]}": bool((val == s2.q[rot_src[k]]).all()) for k, val in snap.items()}
print("stage-1 source wire still holds its bit at stage 2:", alive)
print("""Reading: the low table bits are overwritten before the product exists, so adding them into the product needs
those 8 bits kept alive (8 more qubits, +13%) or recomputed. At stage 1, 61-62 of the 62 wires are busy. The merge
saves at most ~57 Toffoli-eq of rotations (~4%), so on this design it does not pay. A design that phases once from one
accumulator would have to be built around it from the start.""")

### Error budget across x
The worst input uses 99.93% of its tolerance (x = 161), so the rotations get almost no synthesis budget. Every error
source has been squeezed to just under tol(x) = 2^-24 / sqrt(x) turns.

In [ ]:
xs_e = np.unique(np.concatenate([np.arange(1, 3000), np.geomspace(1, 3133452, 6000).astype(np.int64), [161]])).astype(np.int64)
_, se = check(INV, xs_e, seed=2)
target, tol = target_and_tol(xs_e)
ratio = np.abs((se.ph - target).view(np.int64)).astype(float) / tol.astype(float)
fig, ax = plt.subplots(figsize=(9, 2.8))
ax.scatter(xs_e, ratio, s=3, color=C[0], alpha=0.5); ax.set_xscale("log"); ax.axhline(1, color="#52514e", lw=1)
ax.set_title("|phase error| / tolerance for each x (1.0 = fail)"); ax.set_xlabel("x"); ax.set_ylabel("used fraction")
plt.tight_layout(); plt.show()

### Splicing all public 62-qubit circuits: a negative result (and a sampling trap)
HW-25 joined two public circuits where they agree on every wire. We generalised it: a cut is *clean* when the
condition stack is empty and no classical bit written before it is read after it; at a clean cut the rest of the
circuit depends only on the wires, and the phase so far is a function of x alone. Two circuits whose clean cuts hold
the same wires and phase on every sampled input can be joined there. A shortest path over all such sync points of
all public 62-qubit invsqrt circuits (manwithacat's *VQCC public circuits* dataset) gives the cheapest splice.

In [ ]:
def bits_last_read(ops):
    last = {}
    for k, (n, qs, bs, c, a) in enumerate(ops):
        if c is not None: last[c] = k
        if n == "BIT_INVERT": last[bs[0]] = k
    return last
def clean_cuts(path, xs, seed=7):
    ops, regs = parse(path)
    nb = 1 + max(max((max(o[2]) for o in ops if o[2]), default=0), max((o[3] for o in ops if o[3] is not None), default=0))
    last = bits_last_read(ops); s = Sim(ops, 62, nb, len(xs), seed)
    for i, qi in enumerate(regs["r0"]): s.q[qi] = (xs >> i) & 1 == 1
    H = lambda: hashlib.blake2b(np.packbits(s.q).tobytes() + s.ph.tobytes(), digest_size=16).hexdigest()
    cuts, open_until, depth, rot = [(0, H(), 0.0, 0)], -1, 0, 0
    for k, (n, qs, bs, c, a) in enumerate(ops):
        depth += (n == "PUSH_CONDITION") - (n == "POP_CONDITION")
        s.run(k, k + 1)
        for b in bs: open_until = max(open_until, last.get(b, -1))
        if n == "Z_POW" and a % (1 << 61): rot += 1
        if depth == 0 and open_until <= k: cuts.append((k + 1, H(), float(s.toff.mean()), rot))
    return cuts
def splice(data, rot_eq):
    by_hash = collections.defaultdict(list)
    for name, cuts in data.items():
        for i, c in enumerate(cuts): by_hash[c[1]].append((name, i))
    start = next(iter(data.values()))[0][1]; dist, pq, best = {}, [], None
    for node in by_hash[start]: dist[node] = 0.0; heapq.heappush(pq, (0.0, *node))
    while pq:
        d, name, i = heapq.heappop(pq)
        if d > dist.get((name, i), 1e18): continue
        cuts = data[name]
        if i == len(cuts) - 1:
            best = min(best or (1e18,), (d, name)); continue
        w = cuts[i + 1][2] - cuts[i][2] + rot_eq * (cuts[i + 1][3] - cuts[i][3])
        for nxt, nd in [((name, i + 1), d + w)] + [(o, d) for o in by_hash[cuts[i][1]] if o != (name, i)]:
            if nd < dist.get(nxt, 1e18): dist[nxt] = nd; heapq.heappush(pq, (nd, *nxt))
    return best
pub = [p for p in inp.rglob("*.kmb") if "/invsqrt/" in str(p)]
cands = {}
for p in pub:
    c = km.Circuit.from_file(p, format="kmb")
    if c.num_qubits == 62: cands[p.stem[:12]] = to_kmx(p, TMP / f"pub_{p.stem[:12]}.kmx")
log(len(pub), "public invsqrt circuits,", len(cands), "with 62 qubits")
rot_eq = (0.56 * math.log2(1 / v["invsqrt"]["rotation_eps"]) + 5.3) / 4
for label, n_rand in (("small sample (512 shots)", 384), ("large sample (4,098 shots)", 3072)):
    xs_s = np.concatenate([np.arange(1, 1 + (128 if n_rand == 384 else 1024)), np.random.default_rng(11).integers(1, 3133453, n_rand), [161, 3133452]]).astype(np.int64)
    data = {k: clean_cuts(p, xs_s) for k, p in cands.items()}
    singles = {k: c[-1][2] + rot_eq * c[-1][3] for k, c in data.items()}
    best = splice(data, rot_eq); b1 = min(singles, key=singles.get)
    print(f"{label}: best single {b1} {singles[b1]:.3f}, best splice {best[0]:.3f} ({best[0] - singles[b1]:+.3f})")
print("""Reading: on the small sample the splice looks 1.2 Toffolis cheaper; on the large one the gain vanishes (it was the
search picking segments that were cheap on that sample by chance: the conditional blocks fire on random measurement
outcomes). Re-measure any selected candidate on a fresh, larger sample. The public 62-qubit circuits form one lineage
and HW-25 already holds the cheapest pieces at clean cuts.""")
log("splice done")

### The invsqrt landscape: 73 public circuits in qubits x Toffoli-equivalents
Each point is a public circuit measured with kmsim on the same 4,000 inputs; the grey curves are constant score
(Q x C). The frontier moved from 260 qubits down to a dense cluster at 62, and since then mostly along the C axis.

In [ ]:
land = []
xs_l = np.concatenate([np.arange(1, 1001), np.random.default_rng(4).integers(1, 3133453, 3000), [161, 3133452]]).astype(np.int64)
for p in sorted(inp.rglob("*.kmb")):
    if "/invsqrt/" not in str(p): continue
    path = to_kmx(p, TMP / "land.kmx")
    try: r_, s_ = check(path, xs_l, seed=1)
    except Exception: continue
    if not (r_["restored"] and r_["dirty"] == 0 and r_["worst_ratio"] <= 1): continue
    ops_l, _ = parse(path); n_rot = sum(1 for o in ops_l if o[0] == "Z_POW" and o[4] % (1 << 61))
    land.append((p.stem[:8], r_["nq"], r_["toff_mean"] + n_rot * rot1))
L = np.array([(q, c) for _, q, c in land], dtype=float)
fig, ax = plt.subplots(figsize=(9, 3.6))
qq = np.linspace(55, 270, 200)
for sc in (1e5, 2e5, 4e5, 1e6, 2e6):
    ax.plot(qq, sc / qq, color="#d6d5d0", lw=1); ax.text(260, sc / 260, f" {sc:,.0f}", color="#8a8984", fontsize=8, va="center")
ax.scatter(L[:, 0], L[:, 1], s=26, color=C[0], edgecolor="#fcfcfb", linewidth=1.2, zorder=3, label="public circuit")
best_i = int(np.argmin(L[:, 0] * L[:, 1])); ax.scatter(*L[best_i], s=70, color=C[1], zorder=4, label="used here (HW-25)")
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("qubits Q"); ax.set_ylabel("Toffoli-equivalents C")
ax.set_ylim(L[:, 1].min() * 0.85, L[:, 1].max() * 1.5)
ax.set_title("invsqrt: public circuits and lines of equal cost Q x C"); ax.legend(frameon=False, loc="upper right")
plt.tight_layout(); plt.show()
print(f"{len(land)} valid circuits; cheapest Q x C = {L[best_i, 0] * L[best_i, 1]:,.0f} ({land[best_i][0]})")

## 4. DQI: the readout-window identity, and where the decoder is tight
### 4.1 The identity used by our solution (proof by exhaustion)
A *readout window* is `CSWAP c a b` followed by X-measuring both outputs. The CSWAP costs a Toffoli. The same phase
and the same classical record come from
`CX a b; HMR a -> m1; CZ c b if m1; HMR b -> m2; BIT_INVERT m1 if m2`, with no Toffoli:
the phase is m1 a + m1 c (a xor b) + m2 (a xor b) = (m1 xor m2) a' + m2 b' with a' = c ? b : a, b' = c ? a : b.
kmsim checks it on all 8 inputs (c, a, b) and both measurement outcomes, with an extra wire watching the phase.

In [ ]:
before = """CSWAP q0 q1 q2
HMR q1 b0
HMR q2 b1
"""
after = """CX q1 q2
HMR q1 b0
CZ q0 q2 if b0
HMR q2 b1
BIT_INVERT b0 if b1
"""
def run_text(txt, c, a, b, m_seed):
    path = TMP / "gadget.kmx"; path.write_text(txt)
    ops, _ = parse(path); s = Sim(ops, 3, 2, 4, m_seed)
    s.q[0], s.q[1], s.q[2] = c, a, b
    s.run()
    return s.b.copy(), s.ph.copy(), s.q.copy()
ok = True
for c in (0, 1):
    for a in (0, 1):
        for b in (0, 1):
            for seed in range(8):  # the 4 shots per seed cover all (m1, m2) pairs
                b0, p0, q0 = run_text(before, c, a, b, seed); b1_, p1, q1 = run_text(after, c, a, b, seed)
                # same measurement draws are not aligned between the two circuits, so compare the relation
                # phase == (-1)^(record . (a', b')) in each
                ap, bp = (b, a) if c else (a, b)
                for bits_, ph, qq in ((b0, p0, q0), (b1_, p1, q1)):
                    expect = ((bits_[0] & bool(ap)) ^ (bits_[1] & bool(bp))).astype(np.uint64) << np.uint64(63)
                    ok &= bool((ph == expect).all()) and not qq.any()
print("readout-window identity holds on every input and outcome:", ok)

### 4.2 Region profile: what each DQI line pays, by decoder subroutine
The DQI files carry INFO_TAG markers around subroutines. Counting static Toffolis between markers lets us compare
lineages region by region. This is how we found the two regions where the older `e9f54be3`/`9fcda0c1` line was
cheaper than the `cd71c482` line that `bb4287e7` builds on.

In [ ]:
def region_profile(kmx_path, live_trace=False):
    regions, cur, depth = [], {"tag": "<start>", "toff": 0, "ops": 0}, 0
    WRITE = {"X": (1,), "CX": (2,), "CCX": (3,), "SWAP": (1, 2), "CSWAP": (2, 3)}
    zero, cur_live, peak, trace, n = None, 0, 0, [], 0
    for line in open(kmx_path):
        if line.startswith("APPEND_TO_REGISTER"):
            if live_trace and line.split()[1].startswith("q"):
                q = int(line.split()[1][1:]); zero = zero or [True] * 4096
                if zero[q]: zero[q] = False; cur_live += 1
            continue
        if line.startswith("REGISTER"): continue
        if line.startswith("INFO_TAG"):
            regions.append(cur); cur = {"tag": line[9:].strip().strip('"'), "toff": 0, "ops": 0}; continue
        t = line.split(); op = t[0]; n += 1
        depth += (op == "PUSH_CONDITION") - (op == "POP_CONDITION")
        cur["ops"] += 1
        if op in TOFF: cur["toff"] += 1
        if live_trace:
            zero = zero or [True] * 4096
            unc = depth == 0 and "if" not in t
            if op in ("HMR", "R") and unc:
                q = int(t[1][1:])
                if not zero[q]: zero[q] = True; cur_live -= 1
            elif op == "SWAP" and unc:
                a_, b_ = int(t[1][1:]), int(t[2][1:]); zero[a_], zero[b_] = zero[b_], zero[a_]
            elif op in WRITE:
                for i in WRITE[op]:
                    q = int(t[i][1:])
                    if zero[q]: zero[q] = False; cur_live += 1
            if n % 20000 == 0: trace.append((n, cur_live))
            peak = max(peak, cur_live)
    regions.append(cur)
    return regions, trace, peak
dqi_src = {"ours": TMP / "sub" / "dqi.kmb"}
for p in inp.rglob("*.kmb"):
    for w in ("cd71c4822aa0", "9fcda0c1aeb3"):
        if p.name.startswith(w): dqi_src[w[:8]] = p
for z in inp.rglob("submission.zip"):
    if "shared-products" in str(z).lower():
        with zipfile.ZipFile(z) as zz: zz.extract("dqi.kmb", TMP / "bb42")
        dqi_src["bb4287e7"] = TMP / "bb42" / "dqi.kmb"
prof = {}
for k, p in dqi_src.items():
    kx = to_kmx(p, TMP / "dqi_tmp.kmx")
    prof[k] = region_profile(kx, live_trace=(k == "ours"))
    log(k, "regions", len(prof[k][0]), "static Toffolis", sum(r["toff"] for r in prof[k][0]))
by_tag = {k: collections.Counter() for k in prof}
for k, (regs_, _, _) in prof.items():
    for r in regs_: by_tag[k][r["tag"].replace("@exit:", "").replace("@enter:", "")] += r["toff"]
top = [t for t, _ in by_tag["ours"].most_common(10)]
print("Static Toffolis count every gate in the file, including branches that run only for some measurement outcomes;")
print("bb4287e7 and ours carry larger static blocks but execute fewer Toffolis per shot (394,117 vs 394,416 for cd71c482).")
print(f"{'subroutine (static Toffolis)':32s}" + "".join(f"{k:>12s}" for k in prof))
for t in top: print(f"{t[:32]:32s}" + "".join(f"{by_tag[k][t]:12,d}" for k in prof))

### 4.3 Qubits in use along the decoder (static liveness)
A wire is counted from its first write until an unconditional `HMR`/`R` clears it. The decoder hits all 892 wires in
hundreds of short episodes inside the GF(2^12) multipliers, so removing one qubit (worth 0.22% of the DQI cost,
because the cost has Q squared) needs a multiplier that is narrower everywhere, not a local patch.

In [ ]:
_, trace, peak = prof["ours"]
tr = np.array(trace)
fig, ax = plt.subplots(figsize=(9, 2.8))
ax.plot(tr[:, 0] / 1e6, tr[:, 1], color=C[0], lw=1)
ax.axhline(840, color="#52514e", lw=1); ax.text(0, 842, " 840 = syndrome input (cannot shrink)", color="#52514e", va="bottom")
ax.set_title(f"DQI: wires holding data (peak {peak} of 892)"); ax.set_xlabel("operation index (millions)"); ax.set_ylabel("wires")
ax.set_ylim(830, 895); plt.tight_layout(); plt.show()

### 4.4 The rare failures (why the public score of this line is a coin flip)
Every file of the `cd71c482` line, including `bb4287e7` and ours, fails on roughly 6e-6 of the verifier's shots
(an approximate rare branch inherited from the source). The board draws 102,400 shots per file from the file's hash
and a secret salt, so about half of the files of this line draw zero failures (R = 1.1913) and half draw one or more
(R = 1.1967 for one failure, about -0.04 on the score). Measured with the official verifier:

| file | default salt | `s1` | `s2` |
|---|---|---|---|
| `bb4287e7` | 0 | 0 | 1 |
| ours | 1 | 2 | 0 |

Our board draw had zero failures. The private leaderboard draws again. **A decoder of this line that is exact on
every shot would make the score salt-proof**: that is open problem 1 below.

### 4.5 Ablation: what each part of our DQI change does (official verifier)
Four builds on top of `bb4287e7`, each verified with the unchanged official DQI verifier (102,400 shots, default
salt): window region A only (`aligned_route`), region B only (`inverse12 products`), both (our submission), and both
plus Heliosli's retained-product region, which depends on two other regions that `bb4287e7` rewrote.

In [ ]:
import subprocess
kx = {k: to_kmx(p, TMP / f"dqi_{k}.kmx") for k, p in dqi_src.items() if k in ("cd71c482", "9fcda0c1", "bb4287e7")}
def scan_regions(path, keep):
    tags, kept, idx = [""], {}, 0
    for line in open(path):
        if line.startswith(("APPEND_TO_REGISTER", "REGISTER")): continue
        if line.startswith("INFO_TAG"): idx += 1; tags.append(line)
        if idx in keep: kept.setdefault(idx, []).append(line)
    return tags, kept
ALL = [673, 16783, 17409]
tA, RA = scan_regions(kx["cd71c482"], set(ALL)); _, RH = scan_regions(kx["9fcda0c1"], set(ALL)); tB, _ = scan_regions(kx["bb4287e7"], set())
sm = difflib.SequenceMatcher(a=tA, b=tB, autojunk=False)
amap = {blk.a + k: blk.b + k for blk in sm.get_matching_blocks() for k in range(blk.size)}
def build(regions, out):
    jm = {amap[i]: i for i in regions}; idx, done = 0, set()
    with open(out, "w") as f:
        for line in open(kx["bb4287e7"]):
            if line.startswith("INFO_TAG"): idx += 1
            if idx in jm and not line.startswith(("APPEND_TO_REGISTER", "REGISTER")):
                if idx not in done: f.writelines(RH[jm[idx]]); done.add(idx)
                continue
            f.write(line)
    kb = pathlib.Path(str(out).replace(".kmx", ".kmb")); km.Circuit.from_file(out, format="kmx").to_file(kb, format="kmb"); return kb
def verify_dqi(path, salt=""):
    env = dict(os.environ, PYTHONPATH=f"{KM}:{VER}")
    r = subprocess.run([sys.executable, str(VER / "dqi/verify_submission_dqi.py"), str(path), "--salt", salt], capture_output=True, text=True, env=env, cwd=str(VER), timeout=3600)
    return json.loads(r.stdout[r.stdout.index("{"):])
abl = {}
abl["bb4287e7 (base)"] = verify_dqi(dqi_src["bb4287e7"])
for name, regions in (("+ window region A", [673]), ("+ window region B", [16783]), ("+ both (our submission)", [673, 16783]), ("+ both + retained product", [673, 16783, 17409])):
    abl[name] = verify_dqi(build(regions, TMP / "abl.kmx")); log(name, abl[name]["toffoli_count"], abl[name]["value_errors"], abl[name]["phase_errors"])
names = list(abl); tof = [abl[n]["toffoli_count"] for n in names]; pe = [abl[n]["phase_errors"] for n in names]; ve = [abl[n]["value_errors"] for n in names]
print(f"{'build':28s} {'Toffolis/shot':>14s} {'vs base':>9s} {'value err':>10s} {'phase err':>10s}")
for n, t_, a_, b_ in zip(names, tof, ve, pe): print(f"{n:28s} {t_:14,.2f} {t_ - tof[0]:+9.2f} {a_:10d} {b_:10d}")
fig, axs = plt.subplots(1, 2, figsize=(10, 2.8))
axs[0].barh(names, [t_ - tof[0] for t_ in tof], color=[C[0]] * 4 + [C[7]], height=0.55); axs[0].invert_yaxis()
axs[0].set_title("Toffolis per shot vs bb4287e7"); axs[0].axvline(0, color="#52514e", lw=1)
axs[1].barh(names, pe, color=[C[0]] * 4 + [C[7]], height=0.55); axs[1].invert_yaxis(); axs[1].set_yticklabels([])
axs[1].set_title("phase errors in 102,400 shots")
for i, p_ in enumerate(pe): axs[1].text(p_, i, f" {p_}", va="center", color="#0b0b0b")
plt.tight_layout(); plt.show()
print("""Reading: each window saves exactly 24 Toffolis on every shot and the two add up. The retained product saves one
more but needs two companion regions; alone it breaks the phase on ~1.7% of shots, which the verifier prices as a
3x repetition overhead. Cross-lineage transplants must move whole dependency groups, and must be verified.""")

### 4.6 How lucky is a board draw?
The `cd71c482` line fails on rare syndromes. Pooling our official runs (two files, three salts, 614,400 shots, plus
the two new single-region builds above) gives the failure rate; the board's 102,400-shot draw then has a Poisson number of
failures, each of which moves the repetition factor R and the score.

In [ ]:
fails = [0, 0, 1, 1, 2, 0] + [abl[n]["value_errors"] + abl[n]["phase_errors"] for n in names[1:3]]  # new files only (no repeated draws)
eps = sum(fails) / (102400 * len(fails)); lam = eps * 102400
sys.path.insert(0, str(VER / "dqi")); import verify_submission_dqi as vq
probs, scores = [], []
for k in range(5):
    pk = math.exp(-lam) * lam ** k / math.factorial(k)
    R = vq.repetition_overhead(vq.clopper_pearson_upper_bound(k, 102400, -128.0))
    dq = rows["dqi"][2] / 1.1912604648115832 * R
    sc = (REF["invsqrt"] / rows["invsqrt"][2] * REF["dqi"] / dq * REF["dfthcsos"] / rows["dfthcsos"][2]) ** (1 / 3)
    probs.append(pk); scores.append(sc)
fig, ax = plt.subplots(figsize=(7, 2.8))
ax.bar([f"{k} fail" for k in range(5)], probs, color=C[0], width=0.55)
for k, (p_, s_) in enumerate(zip(probs, scores)): ax.text(k, p_, f"{s_:.4f}", ha="center", va="bottom", color="#0b0b0b", fontsize=9)
ax.set_title(f"Board draw: probability of k failed shots (label = score), eps = {eps:.1e}"); ax.set_ylabel("probability")
plt.tight_layout(); plt.show()
print(f"expected failures per draw {lam:.2f}; P(0) = {probs[0]:.2f}; expected score {sum(p * s for p, s in zip(probs, scores)) / sum(probs):.5f}")
print("""Reading: every file of this line has roughly a coin flip between ~25.367 and ~25.33 on each salt; the private
board draws again. Hundreds of portfolios sit inside this band, so expect a large shake-up there, unless a decoder of
the line becomes exact.""")

### 4.7 The DQI landscape: public files in qubits x Toffolis
Cost = R x Q^2 x T, so the grey lines of equal cost are T proportional to 1/Q^2. Two families dominate: the 891-qubit
line and the 892-qubit line that won by trading one qubit for ~2,500 Toffolis.

In [ ]:
dq_pts = []
for r in man:
    if r["task"] != "dqi": continue
    c_ = fnum(r["dqi_zero_failure_cost"]) if r["dqi_zero_failure_cost"] else fnum(r["local_task_score"]); q_ = fnum(r["qubits"])
    if c_ == c_ and q_ == q_ and c_ < 1e12: dq_pts.append((q_, c_ / (1.1912604648115832 * q_ * q_)))
D = np.array(dq_pts); D = D[(D[:, 0] >= 885) & (D[:, 0] <= 900)]
fig, ax = plt.subplots(figsize=(9, 3.4))
x0, x1 = D[:, 0].min() - 1, D[:, 0].max() + 1; y0, y1 = D[:, 1].min() * 0.985, np.percentile(D[:, 1], 85)
qq = np.linspace(x0, x1, 100); Rf = 1.1912604648115832
for sc in (3.75e11, 3.9e11, 4.1e11, 4.4e11):
    ax.plot(qq, sc / (Rf * qq ** 2), color="#d6d5d0", lw=1)
    yr = sc / (Rf * x1 ** 2)
    if y0 < yr < y1: ax.text(x1, yr, f" {sc:.2e}", color="#8a8984", fontsize=8, va="center")
    else:
        xa = math.sqrt(sc / (Rf * y1 * 0.97))
        if x0 < xa < x1: ax.text(xa, y1 * 0.97, f"{sc:.2e} ", color="#8a8984", fontsize=8, ha="right")
ax.set_xlim(x0, x1 + 2.5); ax.set_ylim(y0, y1)
ax.scatter(D[:, 0], D[:, 1], s=22, color=C[0], edgecolor="#fcfcfb", linewidth=1.2, zorder=3, label="public file")
ax.scatter([rows["dqi"][0]], [rows["dqi"][1]], s=70, color=C[1], zorder=4, label="ours")
ax.set_xlabel("qubits Q"); ax.set_ylabel("Toffolis per shot T")
ax.set_title("DQI: public files and lines of equal cost Q^2 x T"); ax.legend(frameon=True, facecolor="#fcfcfb", edgecolor="#fcfcfb", loc="lower right")
plt.tight_layout(); plt.show()

## 5. Chemistry: half of the cost is in the verifier's formula
The chemistry cost is lambda_eff x T x Q over the whole block encoding. The verifier prices the state preparations
and Givens rotations from the factorization shape (R, B, C) and `brot` (`verifiers/dfthcsos/scoreinfra/costs_prepare.py`);
only the ANGLE loaders are simulated.

In [ ]:
sys.path.insert(0, str(VER / "dfthcsos" / "scoreinfra"))
from costs_prepare import SpinFreeData_V1 as Model
res = v["dfthcsos"]["resources"]
print({k: res[k] for k in ("outer_state_prep_toffoli", "inner_state_prep_toffoli", "givens_rotations_toffoli", "angles_toffoli", "persistent_qubits", "temporary_qubits", "angles_temporary_qubits")})
W_shape = np.load(TMP / "sub" / "dfthcsos" / "submission.npz")["W"].shape
print("factorization shape (R, B, C):", W_shape, " brot:", res["brot"])
def model_cost(R, B, C, angles_T=res["angles_toffoli"], ang_pers=res["angles_persistent_qubits"], ang_tmp=res["angles_temporary_qubits"], beta=res["brot"]):
    m = Model(R=R, B=B, C=C, N=76, chi=10, beta=beta)
    o, i = m.outer_state_prep_qubit_cost(), m.inner_state_prep_qubit_cost()
    T = m.outer_state_prep() + m.inner_state_prep() + m.givens_rotations() + angles_T
    return T * (o["persistent"] + i["persistent"] + ang_pers + max(o["temporary"], i["temporary"], ang_tmp))
R0, B0, C0 = W_shape; base_c = model_cost(R0, B0, C0)
Bs = list(range(100, 200)); rel = []
for B in Bs:
    try: rel.append(model_cost(R0, B, C0) / base_c)
    except ValueError: rel.append(np.nan)
fig, ax = plt.subplots(figsize=(9, 2.8))
ax.plot(Bs, rel, color=C[0]); ax.axvline(B0, color=C[1], lw=1); ax.text(B0, np.nanmax(rel), f" current B = {B0}", color="#52514e", va="top")
ax.set_title("Chemistry: T x Q of the cost model vs. basis size B (R, C fixed; accuracy not modelled)")
ax.set_xlabel("B"); ax.set_ylabel("relative T x Q"); plt.tight_layout(); plt.show()
print("""Reading: B = 143 sits at the edge of a step (the inner QROAM block size changes at B = 128/144), and smaller
shapes would cut the formula's cost by up to ~29%, but the encoded Hamiltonian must stay within 20 mHa (both error
gates are already at 19.99998 mHa). The lever is a better factorization at a smaller (R, B, C), a numerical problem,
not circuit surgery.""")

### 5.1 The chemistry landscape: lambda_eff against T x Q
Each public chemistry submission is a point: the factorization sets lambda_eff (and, through the cost model, part of
T and Q), the loaders set the rest. The gains came first from the factorization, then from the loaders.

In [ ]:
ch_pts = [(fnum(r["lambda_eff"]), fnum(r["T_total"]) * fnum(r["Q_total"]), r["release_date"][:10]) for r in man if r["task"] == "chemistry"]
ch_pts = [p for p in ch_pts if p[0] == p[0] and p[1] == p[1]]
P = np.array([(a, b) for a, b, _ in ch_pts]); days = np.array([np.datetime64(d) for *_, d in ch_pts]).astype("datetime64[D]").astype(float)
fig, ax = plt.subplots(figsize=(9, 3.4))
sc_ = ax.scatter(P[:, 1], P[:, 0], c=days, cmap="Blues", s=22, edgecolor="#b9b8b3", linewidth=0.5, zorder=3)
ax.scatter([ch_T * ch_Q], [ch_l], s=80, color=C[1], zorder=4, label="ours (maoxin123 v22 loaders)")
cb = plt.colorbar(sc_, ax=ax, pad=0.01); cb.set_label("release day (darker = later)"); cb.set_ticks([])
ax.set_xscale("log"); ax.set_xlabel("T x Q of the block encoding"); ax.set_ylabel("lambda_eff (Ha)")
ax.set_title("Chemistry: public submissions (cost = lambda_eff x T x Q)"); ax.legend(frameon=False, loc="upper right")
plt.tight_layout(); plt.show()

## 6. Open problems (ideas welcome in the comments)
1. **An exact DQI on the cd71c482 line.** Find the rare branch that fails (~6e-6 of shots) and repair it for fewer
   Toffolis than the score gains from R = 1.1913 on every salt (about +0.04 on the score whenever a draw would fail).
2. **A single phase stream for invsqrt.** 33 rotations instead of 41 are feasible for the margin (a linear program
   over all x); the design has to keep the table's low bits alive or seed the multiplier's accumulator with them.
3. **A narrower GF(2^12) multiplier for DQI.** 892 -> 891 wires is worth -0.22% of the DQI cost; the peak is hit in
   hundreds of places, all inside the multipliers.
4. **A smaller chemistry factorization** within the 20 mHa gates (section 5).
5. **More cross-lineage transplants.** The region profile (4.2) compares any two DQI files of the same structure in a
   few minutes; newer files may again miss an older trick.

If you test one of these, please post what happened, negative results included. We will credit every
contribution in the next version.

## Credits and licences
- **DQI**: Atakan Aldemir (`bb4287e7` and the readout windows of `e9f54be3`); team *i want to run gemma 9b*
  (`b4dc000c`, `cd71c482`); Heliosli (`9fcda0c1`) and the decoder line they credit (Syed Asad Ali, SomeJoint,
  jday96314, Kirill Tymchishin and others).
- **invsqrt** `6c6a734d`: manwithacat (James Barlow, HW-25), gengsr, maoxin123 and the lineage they credit (Syed Asad
  Ali, Hosen42, Kirill Tymchishin, Atakan Aldemir, Ertuğrul Özer, busyaprime, yoyobar, mrmeows). The two-stream
  observation follows Oleg Sher's *Quantum Circuits from Zero* notebook.
- **Chemistry**: maoxin123 (v22 loaders) on ozertuu's table `8afd4c67` and the authors it credits.
- Public circuits: manwithacat's *VQCC public circuits* dataset (Apache-2.0). Tasks, kickmix and verifiers: Google
  Quantum AI / Qualtran.

The code of this notebook (including `kmsim`) is Apache-2.0.

In [ ]:
log("done")